# Model proposal for continous integration

Here we  show the proposal along with the final outupt:

- smooothing using the LOESS filter with a moving window of 7 deltas (L2)

<a id="table"></a>

## Table of Contents

- [Aim of the project](#aim)

0. [Summary of method](#method)

    1. [Create lookup table](#lookptable)
    2. [Download satellite images](#donwload)
    5. [Create zarr folder for historical analysis](#create_zarr)
    6. [Run historical NDVI processing](#historic)

- [Case tested](#case-tested)

    - [Case 1: lowland broadleaf](#lowland-broadleaf)
    - [Case 2: highland broadleaf](#highland-broadleaf)
    - [Case 3: lowland evergreen](#lowland-evergreen)
    - [Case 4: highland broadleaf](#highland-evergreen)
    - [Case 5: fire-affected area](#fire)
    - [Case 6: nearby fire-affected area](#non-fire)
    - [Case 7: 2018 drought-affected area](#drought)
    - [Case 8: Vaia storm-affected area](#storm)

- [Area visualization](#areas)



<a name = "aim"></a>

# Aim of the project

The goal of the project is to create a workflow that automatically process newly acquired NDVI data from Senitnel-2 at 10m of spatial resolution at daily scale. The workflow must be able to differentiate the new data as true observation or outlier. To do so, we have created an outlier detection method based on 2 global parameters. Each data is validated by calculating the difference with the expected value and the actual one, called delta, and the difference between the delta of the observed value to the neighbouring delta, called delta-delta. 

The validation is made with 2 global parameters: the delta threshold and delta-delta threshold, but set at 0.1.


<a name = "method"></a>

# Summary of method

Here, I'll described the method proposed to perform the NDVI processing on the full timeserie from april 2017 to november 2025.



<a name = "lookptable"></a>

## 0_create_lookuptable.py

The analysis includes the donwload of satellite images, the pixel-wise outlier detection, smoothing the observed NDVI and linearly interpolate the missing data between observation at dailiy scale.

To do so, we used the model developed by Samantha, generating the 6 parameters required to calculate the lower and upper double logistic functions. We average the values for each DOY and pixel to create a lookuptable used to perform the analysis.

The lookup table was computed on our machine and the data are transferred at /mnt/data1/UniBe-swiss-ndvi/data/lookup_table_median_ndvi.zarr.

<a name = "donwload"></a>

## 1_extract_swisstopo_dataset.py

To download the satellite images, we use the pystac_client library. We cover the entire Switzerland from 2017-04-01 to 2025-11-30. The data were selected based on the forest mask avaible on Swisstopo VHI dataset.

We apply a filter based on 4 bands, which at least one conditions is met  (green == 9999) | (swir_10m == 9999) (terrain_mask == 255) | (cloud_mask == 255) 

After the filtering, the NDVI and NDSI are computed, the NDVI is filtered out when NDVI >= 0.43. The missing data are flagged with a placeholder of -2^15. The values with no data at given timestep (due to the different orbit) are flagged with a value of 2^16 -1. 

TODO: add forset_mask explanation

Along with the NDVI and NDSI, we retrieve the date of each image, the pixel ID and spatial idx and coordinates, the final output will look like this TODO: add it


## 2_historic_NDVI.py

The second script will perform the NDVI processing oh historical data. The analysis can be split in three parts:

- outlier detection
- anomalies detection and smoothing
- interpolation at dailiy scale

The outlier detection is the first step to filter out the non-observation values and outliers.

The scaled-down observation must be within 0-1 so it is easy to filter out the missing data with -2^15 or no data with 2^16 -1.

After the first filter is applied, we remove the outliers, we defined an outlier according to this defintion:

- The difference between the absolute NDVI value and the corresponded expected value (hereinafter called median) is above a a threshold (0.1), so called delta.
- The difference between the current analysed delta and the two neighbourh delta is above a threshold (0.1), so called delta-delta.

When both conditions are met, the value is flagged as outlier and is removed from the timeserie.

The following passage is to create the delta timeserie used to linearly interpolate the missing data. This array is created by evaluating each delta on a rolling window of 7 values. 

Within the window, we check if the data inside the window are close to the boundaries conditions or have extreme negative NDVI values, if one (or both) conditions are met the delta is added as it is, otherwise the smoothing is performed. 

the smoothing of the deltas is performed on the non-outlier observed NDVI. We use the LOESS alogorithm to perform the smoothing on a rolling window of 7 observation and 3 iterations of the algoritm. 

After the rolling window reach the last-fourth observation (so that is centered) we cannot proceed using this method, hence we append the remaining deltas that will be flagged as "observation yet to smooth" (L1 linearly interpolated product).

After the delta timeserie is created, we linearly interpolate the results by taking into account their position on the timeserie, the interpolated delta are summed to the medians NDVI to obtain the processed NDVI values.

The final timeserie will have from the third to the last fourth observation the smoothing values (L2) and from the day after the alst fourth observation onwards the linarly delta interpolation (L1).

After the processing, we create the mask array for the TIFF generation. The mask will have integer values from 0 to 4 according to this list

- **0**: the data is not an observation and is yet to be smoothed
- **1**: the data is not an observation and is smoothed
- **2**: the data is an observation and is yet to be smoothed
- **3**: the data is an observation and is smoothed
- **4**: the data is an observation and is an outlier

## 3 TIFF generation

The TIFF are generated ...

<a id="case-tested"></a>

[Go back to the table](#table)

## Case tested

We test this model on different biomes and known cases. Each case is represented by 25 pixels. All pixels are collected and arranged according to the following table. 

For each coordinate listed below, we select a square of 60 meters centered aorund the coordinates and we select 25 pixel from it

| Biome                                 | Coordinates (x, y)        | Pixel Range |
|---------------------------------------|---------------------------|-------------|
| Lowland broadleaf                     | 2694491.82, 1126023.20    | 0–24        |
| Highland broadleaf                    | 2692020.28, 1121443.47    | 25–49       |
| Lowland evergreen                     | 2761097.61, 1194613.45    | 50–74       |
| Highland evergreen                    | 2781537.00, 1182975.00    | 75–99       |
| Biscth fire affected area             | 2644029.37, 1134128.20    | 100–124     |
| Biscth fire nearby non-affected area  | 2644328.07, 1134342.81    | 125–149     |
| Drought-affected area                 | 2690025.48, 1287413.03    | 150–174     |
| Vaia storm affected area              | 2689564.74, 1154411.88    | 175–199     |

For each case, we will analyse 
- a pixel in detail, covireing all three products and the latency (difference between current date and smoothing)
- the smoothign of 24 pixels
- the area comparison between raw data and smoothing


In [10]:
# This is just to embed the web page and the videos
from IPython.display import IFrame, Video

<a id="lowland-broadleaf"></a>

[Go back to the table](#table)

## Case 1: lowland broadleaf

The broadleaf biomes selected are nicely represent and do not present any complication.

In [11]:
# area location
x, y = 2694491.82, 1126023.20

# Construct the URL with your coordinates as center
url = f"https://map.geo.admin.ch/#/map?lang=de&center={x},{y}&z=10&topic=ech&layers=ch.swisstopo.zeitreihen@year=1864,f;ch.bfs.gebaeude_wohnungs_register,f;ch.bav.haltestellen-oev,f;ch.swisstopo.swisstlm3d-wanderwege,f;ch.vbs.schiessanzeigen,f;ch.astra.wanderland-sperrungen_umleitungen,f&bgLayer=ch.swisstopo.swissimage"
# Display map in notebook
IFrame(url, width=1000, height=600)

The lowland broadleaf area appears to be correctly smoothed.

There are no significant problem in the outlier detection and smoothing.

On winter 2024, we identified a reduction of NDVI across all points which is correctly captured.

<figure>
  <img src="fig/all_low_broad_1.png">
</figure>


<figure>
  <img src="fig/all_low_broad_2.png">
</figure>

<a id="highland-broadleaf"></a>

[Go back to the table](#table)

## Case 2: Highland broadleaf

In [12]:
# area location
x, y = 2692020.28, 1121443.47

# Construct the URL with your coordinates as center
url = f"https://map.geo.admin.ch/#/map?lang=de&center={x},{y}&z=10&topic=ech&layers=ch.swisstopo.zeitreihen@year=1864,f;ch.bfs.gebaeude_wohnungs_register,f;ch.bav.haltestellen-oev,f;ch.swisstopo.swisstlm3d-wanderwege,f;ch.vbs.schiessanzeigen,f;ch.astra.wanderland-sperrungen_umleitungen,f&bgLayer=ch.swisstopo.swissimage"
# Display map in notebook
IFrame(url, width=1000, height=600)

The highland broadlead biomes appears to have almost no problem in detecting early summer NDVI values.

The lower and upper bands appear different between neighbouring pixels, especially during winter.

We are able to correctly identify and smooth the NDVI timeserie despite the scattering of data.

<figure>
  <img src="fig/all_high_broad_1.png">
</figure>


<figure>
  <img src="fig/all_high_broad_2.png">
</figure>

<a id="lowland-evergreen"></a>

[Go back to the table](#table)

## Case 3: Lowland evergreen

The lowland evergreen biome selected has high variability (as expcted). With the new set of parameters we are able to represent correctly.

In [13]:
# area location
x, y = 2761097.61, 1194613.45

# Construct the URL with your coordinates as center
url = f"https://map.geo.admin.ch/#/map?lang=de&center={x},{y}&z=10&topic=ech&layers=ch.swisstopo.zeitreihen@year=1864,f;ch.bfs.gebaeude_wohnungs_register,f;ch.bav.haltestellen-oev,f;ch.swisstopo.swisstlm3d-wanderwege,f;ch.vbs.schiessanzeigen,f;ch.astra.wanderland-sperrungen_umleitungen,f&bgLayer=ch.swisstopo.swissimage"
# Display map in notebook
IFrame(url, width=1000, height=600)

The lowland evergreen selected area is composed by sparce vegetation and the canopy does not fully cover the ground. This may impact the NDVI values increasing the scattering of them.

There is no much variation between summer and winter NDVI values, which is expected considering that the area selected is a evergreen forset.

There is a sharp drop on winter on the upper and lower bands. This is caused by the fact that the first and last day of the double sigmoid function do not necessarly overlap. 

<figure>
  <img src="fig/all_low_ever_1.png">
</figure>


<figure>
  <img src="fig/all_low_ever_2.png">
</figure>

<a id="highland-evergreen"></a>

[Go back to the table](#table)

## Case 4: Highland evergreen

With the new set of parameters the interquantile range is very large during winter. For that reason we observe a sharp drop during that season. However, LOESS smoothing method has the ability to drastically smooth an extreme value (that we would flag if it was outside the IQR).

In [14]:
# area location
x, y = 2781537.00, 1182975.00 

# Construct the URL with your coordinates as center
url = f"https://map.geo.admin.ch/#/map?lang=de&center={x},{y}&z=10&topic=ech&layers=ch.swisstopo.zeitreihen@year=1864,f;ch.bfs.gebaeude_wohnungs_register,f;ch.bav.haltestellen-oev,f;ch.swisstopo.swisstlm3d-wanderwege,f;ch.vbs.schiessanzeigen,f;ch.astra.wanderland-sperrungen_umleitungen,f&bgLayer=ch.swisstopo.swissimage"
# Display map in notebook
IFrame(url, width=1000, height=600)

In the evergreen biomes there is some scattering but we are able to follow the NDVI timeserie correctly

<figure>
  <img src="fig/all_high_ever_1.png">
</figure>


<figure>
  <img src="fig/all_high_ever_2.png">
</figure>

<a id="fire"></a>

[Go back to the table](#table)

## Case 5: Fire-affected area

With the new set of parameters we are able to correctly flag the drastic reduction in NDVI caused by the fire event. After the event we use the absoulte NDVI to smooth the values instead of the deltas because it is not garantee that the vegetation follows the expected behavior after this drastic event.

In [15]:
# area location
x, y = 2644029.37, 1134128.20 

# Construct the URL with your coordinates as center
url = f"https://map.geo.admin.ch/#/map?lang=de&center={x},{y}&z=10&topic=ech&layers=ch.swisstopo.zeitreihen@year=1864,f;ch.bfs.gebaeude_wohnungs_register,f;ch.bav.haltestellen-oev,f;ch.swisstopo.swisstlm3d-wanderwege,f;ch.vbs.schiessanzeigen,f;ch.astra.wanderland-sperrungen_umleitungen,f&bgLayer=ch.swisstopo.swissimage"
# Display map in notebook
IFrame(url, width=1000, height=600)

<figure>
  <img src="fig/all_fire_1.png">
</figure>


<figure>
  <img src="fig/all_fire_2.png">
</figure>

<a id="non-fire"></a>

[Go back to the table](#table)

## Case 6: Nearby fire-affected area

The nreaby non affected area by the Bistch fire is clearly different and we (rightfully) do not see the fire event.

In [16]:
# area location
x, y = 2644328.07, 1134342.81

# Construct the URL with your coordinates as center
url = f"https://map.geo.admin.ch/#/map?lang=de&center={x},{y}&z=10&topic=ech&layers=ch.swisstopo.zeitreihen@year=1864,f;ch.bfs.gebaeude_wohnungs_register,f;ch.bav.haltestellen-oev,f;ch.swisstopo.swisstlm3d-wanderwege,f;ch.vbs.schiessanzeigen,f;ch.astra.wanderland-sperrungen_umleitungen,f&bgLayer=ch.swisstopo.swissimage"
# Display map in notebook
IFrame(url, width=1000, height=600)


Due to the sparce vege


<figure>
  <img src="fig/all_non_fire_1.png">
</figure>


<figure>
  <img src="fig/all_non_fire_2.png">
</figure>

<a id="drought"></a>

[Go back to the table](#table)

## Case 7: Drought affected area

We selected an area north of Schaffausen as illustrated in Fig. 3 in https://onlinelibrary.wiley.com/doi/10.1111/gcb.15360

This drought event mildy affected the vegetation but is possible to see a sharper drop at the summer 2018 season. We are able to correctly indentified the drought and non drought pixels.

In [17]:
# area location
x, y = 2690025.48, 1287413.03

# Construct the URL with your coordinates as center
url = f"https://map.geo.admin.ch/#/map?lang=de&center={x},{y}&z=10&topic=ech&layers=ch.swisstopo.zeitreihen@year=1864,f;ch.bfs.gebaeude_wohnungs_register,f;ch.bav.haltestellen-oev,f;ch.swisstopo.swisstlm3d-wanderwege,f;ch.vbs.schiessanzeigen,f;ch.astra.wanderland-sperrungen_umleitungen,f&bgLayer=ch.swisstopo.swissimage"
# Display map in notebook
IFrame(url, width=1000, height=600)

<figure>
  <img src="fig/all_drought_1.png">
</figure>


<figure>
  <img src="fig/all_drought_2.png">
</figure>

<a id="storm"></a>

[Go back to the table](#table)

## Case 8: Storm Vaia affected area

The effect of Vaia storm is clearly visible and the NDVI is correctly flagged. It is super interesting to see the different NDVI series after the Vaia storm. The model correctly not flagged any of this different response despite using a nearly identical set of parameters.

The sparce vegetation here again influence the NDVI scattering.

In [18]:
# area location
x, y = 2689564.74, 1154411.88

# Construct the URL with your coordinates as center
url = f"https://map.geo.admin.ch/#/map?lang=de&center={x},{y}&z=10&topic=ech&layers=ch.swisstopo.zeitreihen@year=1864,f;ch.bfs.gebaeude_wohnungs_register,f;ch.bav.haltestellen-oev,f;ch.swisstopo.swisstlm3d-wanderwege,f;ch.vbs.schiessanzeigen,f;ch.astra.wanderland-sperrungen_umleitungen,f&bgLayer=ch.swisstopo.swissimage"
# Display map in notebook
IFrame(url, width=1000, height=600)

<figure>
  <img src="fig/all_storm_1.png">
</figure>


<figure>
  <img src="fig/all_storm_2.png">
</figure>